In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking and good BC acquisition start
        df = df[df['Non_Standard_Braking'] == 0]
        # df = df[df['BC_BadStart'] == 0]
        
        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'Dati(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source
        df['Malfunction'] = 0

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base, df_data

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_Dati01.csv',
    'TestBrakefinal_data_Dati06.csv',
    'TestBrakefinal_data_Dati27.csv'
]

[df, df_monorail] = load_data(model_path, monorail_paths)
df['Malfunction'] = df['Malfunction'].astype(str)
print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

In [ ]:
df_monorail.head()

# Initial Data Analysis 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# 1. OVERALL HEALTH MONITORING ANALYSIS
# ============================================================================

def generate_comprehensive_health_report(df):
    """Generate complete health monitoring report"""
    
    print("="*80)
    print("FREIGHT WAGON BRAKING SYSTEM - HEALTH MONITORING REPORT")
    print("="*80)
    
    # Basic statistics
    total_records = len(df)
    total_wagons = df['BC_ID'].nunique()
    
    print(f"\n📊 DATASET OVERVIEW")
    print(f"{'='*80}")
    print(f"Total Records: {total_records:,}")
    print(f"Total Unique Wagons: {total_wagons}")
    print(f"Data Source Distribution: {df['Source'].value_counts().to_dict()}")
    
    # Overall system health
    healthy_records = (df['Malfunction'] == 0).sum()
    health_rate = (healthy_records / total_records) * 100
    malfunction_rate = 100 - health_rate
    
    print(f"\n🏥 OVERALL SYSTEM HEALTH")
    print(f"{'='*80}")
    print(f"Healthy Records: {healthy_records:,} ({health_rate:.2f}%)")
    print(f"Records with Malfunctions: {total_records - healthy_records:,} ({malfunction_rate:.2f}%)")
    
    # Error analysis
    error_columns = ['Non_Standard_Braking', 'GPS_SensorError', 'SV_Error', 
                     'MBP_Sensor_error', 'MBP_braketiming_error', 
                     'UB_Error', 'UR_Error', 'WV_SensorError']
    
    print(f"\n⚠️  ERROR ANALYSIS")
    print(f"{'='*80}")
    print(f"{'Error Type':<30} {'Count':<10} {'Rate (%)':<10}")
    print(f"{'-'*80}")
    
    error_summary = {}
    for col in error_columns:
        count = df[col].sum()
        rate = (count / total_records) * 100
        error_summary[col] = {'count': count, 'rate': rate}
        print(f"{col:<30} {count:<10} {rate:<10.2f}")
    
    # Leakage analysis
    print(f"\n💧 LEAKAGE ANALYSIS")
    print(f"{'='*80}")
    leakage_counts = df['LeakageLabel'].value_counts()
    for label, count in leakage_counts.items():
        rate = (count / total_records) * 100
        print(f"{label:<30} {count:<10} ({rate:.2f}%)")
    
    # Wagon-specific analysis
    wagons_with_issues = df[df['Malfunction'] == 1]['BC_ID'].nunique()
    print(f"\n🚂 WAGON-LEVEL ANALYSIS")
    print(f"{'='*80}")
    print(f"Wagons with Issues: {wagons_with_issues} out of {total_wagons}")
    print(f"Healthy Wagons: {total_wagons - wagons_with_issues}")
    
    return error_summary

# ============================================================================
# 2. WAGON-LEVEL DETAILED ANALYSIS
# ============================================================================

def wagon_level_analysis(df):
    """Analyze performance by individual wagon"""
    
    print(f"\n\n🔍 DETAILED WAGON ANALYSIS")
    print(f"{'='*80}")
    
    wagon_stats = df.groupby('BC_ID').agg({
        'Non_Standard_Braking': 'sum',
        'GPS_SensorError': 'sum',
        'SV_Error': 'sum',
        'MBP_Sensor_error': 'sum',
        'Malfunction': 'sum',
        'BC_MaxPressure': ['mean', 'std', 'min', 'max'],
        'Brake_power_delay': ['mean', 'std'],
        'Brake_timing_cyl': 'mean',
        'Brake_timing_pipe': 'mean'
    }).round(3)
    
    # Flatten column names
    wagon_stats.columns = ['_'.join(col).strip() for col in wagon_stats.columns.values]
    
    # Add total error count
    wagon_stats['Total_Errors'] = (
        wagon_stats['Non_Standard_Braking_sum'] + 
        wagon_stats['GPS_SensorError_sum'] + 
        wagon_stats['SV_Error_sum'] + 
        wagon_stats['MBP_Sensor_error_sum']
    )
    
    # Sort by total errors (most problematic first)
    wagon_stats_sorted = wagon_stats.sort_values('Total_Errors', ascending=False)
    
    print("\nTop 10 Wagons with Most Issues:")
    print(wagon_stats_sorted[['Total_Errors', 'Malfunction_sum', 
                               'Non_Standard_Braking_sum', 'GPS_SensorError_sum']].head(10))
    
    return wagon_stats

# ============================================================================
# 3. PERFORMANCE METRICS ANALYSIS
# ============================================================================

def performance_analysis(df):
    """Analyze braking performance metrics"""
    
    print(f"\n\n⚡ PERFORMANCE METRICS ANALYSIS")
    print(f"{'='*80}")
    
    # Compare healthy vs malfunction records
    healthy_df = df[df['Malfunction'] == 0]
    malfunction_df = df[df['Malfunction'] == 1]
    
    metrics = ['BC_MaxPressure', 'Brake_energy_cyl', 'Brake_energy_pipe',
               'Brake_power_cyl', 'Brake_power_pipe', 'Brake_timing_cyl', 
               'Brake_timing_pipe', 'Brake_power_delay']
    
    print(f"\n{'Metric':<25} {'Healthy Mean':<15} {'Malfunction Mean':<20} {'Difference':<15}")
    print(f"{'-'*80}")
    
    performance_comparison = {}
    for metric in metrics:
        healthy_mean = healthy_df[metric].mean()
        mal_mean = malfunction_df[metric].mean() if len(malfunction_df) > 0 else 0
        diff = mal_mean - healthy_mean
        performance_comparison[metric] = {
            'healthy': healthy_mean,
            'malfunction': mal_mean,
            'difference': diff
        }
        print(f"{metric:<25} {healthy_mean:<15.3f} {mal_mean:<20.3f} {diff:<15.3f}")
    
    return performance_comparison

# ============================================================================
# 4. ERROR RATE CALCULATION
# ============================================================================

def calculate_error_rates(df):
    """Calculate comprehensive error rates"""
    
    total_records = len(df)
    
    error_rates = {
        'Non_Standard_Braking_Rate': (df['Non_Standard_Braking'].sum() / total_records) * 100,
        'GPS_SensorError_Rate': (df['GPS_SensorError'].sum() / total_records) * 100,
        'SV_Error_Rate': (df['SV_Error'].sum() / total_records) * 100,
        'MBP_Sensor_Error_Rate': (df['MBP_Sensor_error'].sum() / total_records) * 100,
        'MBP_Braketiming_Error_Rate': (df['MBP_braketiming_error'].sum() / total_records) * 100,
        'UB_Error_Rate': (df['UB_Error'].sum() / total_records) * 100,
        'UR_Error_Rate': (df['UR_Error'].sum() / total_records) * 100,
        'WV_Sensor_Error_Rate': (df['WV_SensorError'].sum() / total_records) * 100,
        'Overall_Malfunction_Rate': (df['Malfunction'].sum() / total_records) * 100
    }
    
    return error_rates

# ============================================================================
# 5. VISUALIZATION FUNCTIONS
# ============================================================================

def create_visualizations(df, error_summary, wagon_stats, performance_comparison):
    """Create all visualization plots"""
    
    # Create a large figure with multiple subplots
    fig = plt.figure(figsize=(20, 12))
    
    # -------------------------------------------------------------------------
    # Plot 1: Overall System Health Pie Chart
    # -------------------------------------------------------------------------
    ax1 = plt.subplot(2, 3, 1)
    healthy_count = (df['Malfunction'] == 0).sum()
    malfunction_count = (df['Malfunction'] == 1).sum()
    
    colors = ['#2ecc71', '#e74c3c']
    explode = (0.05, 0)
    
    ax1.pie([healthy_count, malfunction_count], 
            labels=['Healthy', 'Malfunction'],
            autopct='%1.1f%%',
            colors=colors,
            explode=explode,
            startangle=90,
            textprops={'fontsize': 12, 'weight': 'bold'})
    ax1.set_title('Overall System Health Status', fontsize=14, weight='bold', pad=20)
    
    # -------------------------------------------------------------------------
    # Plot 2: Error Type Frequency Bar Chart
    # -------------------------------------------------------------------------
    ax2 = plt.subplot(2, 3, 2)
    error_types = list(error_summary.keys())
    error_counts = [error_summary[key]['count'] for key in error_types]
    
    # Shorten labels for better display
    short_labels = [label.replace('_', '\n') for label in error_types]
    
    bars = ax2.bar(range(len(error_types)), error_counts, color='coral', alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Error Type', fontsize=11, weight='bold')
    ax2.set_ylabel('Count', fontsize=11, weight='bold')
    ax2.set_title('Error Frequency by Type', fontsize=14, weight='bold', pad=20)
    ax2.set_xticks(range(len(error_types)))
    ax2.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
    ax2.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9, weight='bold')
    
    # -------------------------------------------------------------------------
    # Plot 3: Leakage Distribution
    # -------------------------------------------------------------------------
    ax3 = plt.subplot(2, 3, 3)
    leakage_counts = df['LeakageLabel'].value_counts()
    
    bars = ax3.barh(leakage_counts.index, leakage_counts.values, 
                    color='skyblue', alpha=0.7, edgecolor='black')
    ax3.set_xlabel('Count', fontsize=11, weight='bold')
    ax3.set_ylabel('Leakage Status', fontsize=11, weight='bold')
    ax3.set_title('Leakage Distribution', fontsize=14, weight='bold', pad=20)
    ax3.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, (idx, val) in enumerate(leakage_counts.items()):
        ax3.text(val, i, f' {val}', va='center', fontsize=10, weight='bold')
    
    # -------------------------------------------------------------------------
    # Plot 4: Top 10 Problematic Wagons
    # -------------------------------------------------------------------------
    ax4 = plt.subplot(2, 3, 4)
    top_wagons = wagon_stats.nlargest(10, 'Total_Errors')
    
    bars = ax4.bar(range(len(top_wagons)), top_wagons['Total_Errors'], 
                   color='tomato', alpha=0.7, edgecolor='black')
    ax4.set_xlabel('Wagon ID', fontsize=11, weight='bold')
    ax4.set_ylabel('Total Errors', fontsize=11, weight='bold')
    ax4.set_title('Top 10 Wagons with Most Errors', fontsize=14, weight='bold', pad=20)
    ax4.set_xticks(range(len(top_wagons)))
    ax4.set_xticklabels(top_wagons.index, rotation=45, ha='right', fontsize=9)
    ax4.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9, weight='bold')
    
    # -------------------------------------------------------------------------
    # Plot 5: Brake Pressure Distribution (Healthy vs Malfunction)
    # -------------------------------------------------------------------------
    ax5 = plt.subplot(2, 3, 5)
    
    healthy_pressure = df[df['Malfunction'] == 0]['BC_MaxPressure']
    malfunction_pressure = df[df['Malfunction'] == 1]['BC_MaxPressure']
    
    box_data = [healthy_pressure, malfunction_pressure]
    bp = ax5.boxplot(box_data, labels=['Healthy', 'Malfunction'],
                     patch_artist=True, showmeans=True)
    
    # Color the boxes
    colors_box = ['lightgreen', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax5.set_ylabel('Max Pressure', fontsize=11, weight='bold')
    ax5.set_title('Brake Pressure Distribution\n(Healthy vs Malfunction)', 
                  fontsize=14, weight='bold', pad=20)
    ax5.grid(axis='y', alpha=0.3)
    
    # -------------------------------------------------------------------------
    # Plot 6: Brake Timing Comparison
    # -------------------------------------------------------------------------
    ax6 = plt.subplot(2, 3, 6)
    
    healthy_timing_cyl = df[df['Malfunction'] == 0]['Brake_timing_cyl']
    healthy_timing_pipe = df[df['Malfunction'] == 0]['Brake_timing_pipe']
    mal_timing_cyl = df[df['Malfunction'] == 1]['Brake_timing_cyl']
    mal_timing_pipe = df[df['Malfunction'] == 1]['Brake_timing_pipe']
    
    box_data_timing = [healthy_timing_cyl, healthy_timing_pipe, 
                       mal_timing_cyl, mal_timing_pipe]
    bp_timing = ax6.boxplot(box_data_timing, 
                            labels=['Healthy\nCylinder', 'Healthy\nPipe',
                                   'Malfunction\nCylinder', 'Malfunction\nPipe'],
                            patch_artist=True, showmeans=True)
    
    # Color the boxes
    colors_timing = ['lightgreen', 'lightgreen', 'lightcoral', 'lightcoral']
    for patch, color in zip(bp_timing['boxes'], colors_timing):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax6.set_ylabel('Timing (seconds)', fontsize=11, weight='bold')
    ax6.set_title('Brake Timing Comparison', fontsize=14, weight='bold', pad=20)
    ax6.grid(axis='y', alpha=0.3)
    plt.setp(ax6.xaxis.get_majorticklabels(), rotation=0, fontsize=9)
    
    plt.tight_layout()
    plt.savefig('freight_wagon_health_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✅ Visualization saved as 'freight_wagon_health_analysis.png'")
    plt.show()

# ============================================================================
# 6. ADDITIONAL VISUALIZATIONS - TIME SERIES & TRENDS
# ============================================================================

def create_additional_visualizations(df):
    """Create additional detailed visualizations"""
    
    fig = plt.figure(figsize=(20, 10))
    
    # -------------------------------------------------------------------------
    # Plot 1: Error Rate Comparison
    # -------------------------------------------------------------------------
    ax1 = plt.subplot(2, 3, 1)
    error_rates = calculate_error_rates(df)
    
    error_names = list(error_rates.keys())
    error_values = list(error_rates.values())
    
    # Sort by rate
    sorted_indices = np.argsort(error_values)[::-1]
    sorted_names = [error_names[i] for i in sorted_indices]
    sorted_values = [error_values[i] for i in sorted_indices]
    
    short_names = [name.replace('_Rate', '').replace('_', '\n') for name in sorted_names]
    
    bars = ax1.bar(range(len(sorted_names)), sorted_values, 
                   color='steelblue', alpha=0.7, edgecolor='black')
    ax1.set_xlabel('Error Type', fontsize=11, weight='bold')
    ax1.set_ylabel('Error Rate (%)', fontsize=11, weight='bold')
    ax1.set_title('Error Rates Comparison', fontsize=14, weight='bold', pad=20)
    ax1.set_xticks(range(len(sorted_names)))
    ax1.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}%',
                    ha='center', va='bottom', fontsize=8, weight='bold')
    
    # -------------------------------------------------------------------------
    # Plot 2: Brake Energy Comparison (Cylinder vs Pipe)
    # -------------------------------------------------------------------------
    ax2 = plt.subplot(2, 3, 2)
    
    ax2.scatter(df['Brake_energy_pipe'], df['Brake_energy_cyl'], 
               c=df['Malfunction'], cmap='RdYlGn_r', alpha=0.6, s=30, edgecolors='black', linewidth=0.5)
    ax2.plot([df['Brake_energy_pipe'].min(), df['Brake_energy_pipe'].max()],
            [df['Brake_energy_pipe'].min(), df['Brake_energy_pipe'].max()],
            'k--', alpha=0.5, linewidth=2, label='Ideal (Equal)')
    ax2.set_xlabel('Brake Energy Pipe', fontsize=11, weight='bold')
    ax2.set_ylabel('Brake Energy Cylinder', fontsize=11, weight='bold')
    ax2.set_title('Brake Energy: Cylinder vs Pipe', fontsize=14, weight='bold', pad=20)
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # -------------------------------------------------------------------------
    # Plot 3: Brake Power Delay Distribution
    # -------------------------------------------------------------------------
    ax3 = plt.subplot(2, 3, 3)
    
    ax3.hist(df[df['Malfunction']==0]['Brake_power_delay'], bins=30, 
            alpha=0.6, label='Healthy', color='green', edgecolor='black')
    ax3.hist(df[df['Malfunction']==1]['Brake_power_delay'], bins=30, 
            alpha=0.6, label='Malfunction', color='red', edgecolor='black')
    ax3.set_xlabel('Brake Power Delay', fontsize=11, weight='bold')
    ax3.set_ylabel('Frequency', fontsize=11, weight='bold')
    ax3.set_title('Brake Power Delay Distribution', fontsize=14, weight='bold', pad=20)
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    # -------------------------------------------------------------------------
    # Plot 4: Wagon Performance Summary
    # -------------------------------------------------------------------------
    ax4 = plt.subplot(2, 3, 4)
    
    wagon_malfunction_count = df.groupby('BC_ID')['Malfunction'].sum().sort_values(ascending=False)
    
    ax4.bar(range(len(wagon_malfunction_count)), wagon_malfunction_count.values,
           color='orange', alpha=0.7, edgecolor='black')
    ax4.set_xlabel('Wagon Index', fontsize=11, weight='bold')
    ax4.set_ylabel('Malfunction Count', fontsize=11, weight='bold')
    ax4.set_title('Malfunction Count per Wagon', fontsize=14, weight='bold', pad=20)
    ax4.grid(axis='y', alpha=0.3)
    
    # -------------------------------------------------------------------------
    # Plot 5: Multiple Error Types by Wagon
    # -------------------------------------------------------------------------
    ax5 = plt.subplot(2, 3, 5)
    
    error_by_wagon = df.groupby('BC_ID')[['Non_Standard_Braking', 'GPS_SensorError', 
                                           'SV_Error', 'MBP_Sensor_error']].sum()
    
    top_10_error_wagons = error_by_wagon.sum(axis=1).nlargest(10)
    top_10_data = error_by_wagon.loc[top_10_error_wagons.index]
    
    x = np.arange(len(top_10_data))
    width = 0.2
    
    ax5.bar(x - 1.5*width, top_10_data['Non_Standard_Braking'], width, 
           label='Non Standard', color='red', alpha=0.7)
    ax5.bar(x - 0.5*width, top_10_data['GPS_SensorError'], width, 
           label='GPS Error', color='blue', alpha=0.7)
    ax5.bar(x + 0.5*width, top_10_data['SV_Error'], width, 
           label='SV Error', color='green', alpha=0.7)
    ax5.bar(x + 1.5*width, top_10_data['MBP_Sensor_error'], width, 
           label='MBP Sensor', color='orange', alpha=0.7)
    
    ax5.set_xlabel('Wagon ID', fontsize=11, weight='bold')
    ax5.set_ylabel('Error Count', fontsize=11, weight='bold')
    ax5.set_title('Top 10 Wagons: Error Type Breakdown', fontsize=14, weight='bold', pad=20)
    ax5.set_xticks(x)
    ax5.set_xticklabels(top_10_data.index, rotation=45, ha='right', fontsize=9)
    ax5.legend(fontsize=9)
    ax5.grid(axis='y', alpha=0.3)
    
    # -------------------------------------------------------------------------
    # Plot 6: Pressure Statistics by Wagon
    # -------------------------------------------------------------------------
    ax6 = plt.subplot(2, 3, 6)
    
    wagon_pressure_stats = df.groupby('BC_ID')['BC_MaxPressure'].agg(['mean', 'std'])
    top_10_pressure = wagon_pressure_stats.nlargest(10, 'std')
    
    x_pos = np.arange(len(top_10_pressure))
    ax6.bar(x_pos, top_10_pressure['mean'], yerr=top_10_pressure['std'],
           color='purple', alpha=0.7, edgecolor='black', capsize=5)
    ax6.set_xlabel('Wagon ID', fontsize=11, weight='bold')
    ax6.set_ylabel('Max Pressure (Mean ± Std)', fontsize=11, weight='bold')
    ax6.set_title('Top 10 Wagons: Pressure Variability', fontsize=14, weight='bold', pad=20)
    ax6.set_xticks(x_pos)
    ax6.set_xticklabels(top_10_pressure.index, rotation=45, ha='right', fontsize=9)
    ax6.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('freight_wagon_detailed_analysis.png', dpi=300, bbox_inches='tight')
    print("✅ Detailed visualization saved as 'freight_wagon_detailed_analysis.png'")
    plt.show()

# ============================================================================
# 7. MAIN EXECUTION FUNCTION
# ============================================================================

def run_complete_analysis(df):
    """Run all analysis and create all visualizations"""
    
    print("\n" + "="*80)
    print("STARTING COMPREHENSIVE FREIGHT WAGON HEALTH ANALYSIS")
    print("="*80 + "\n")
    
    # Run all analyses
    error_summary = generate_comprehensive_health_report(df)
    wagon_stats = wagon_level_analysis(df)
    performance_comparison = performance_analysis(df)
    
    # Create visualizations
    print("\n\n📈 GENERATING VISUALIZATIONS...")
    print("="*80)
    create_visualizations(df, error_summary, wagon_stats, performance_comparison)
    create_additional_visualizations(df)
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    print("\nGenerated Files:")
    print("  1. freight_wagon_health_analysis.png - Main health dashboard")
    print("  2. freight_wagon_detailed_analysis.png - Detailed metrics")
    print("\nYou can now review the visualizations and printed analysis above.")
    
    return {
        'error_summary': error_summary,
        'wagon_stats': wagon_stats,
        'performance_comparison': performance_comparison
    }

# ============================================================================
# USAGE EXAMPLE
# ============================================================================

# Run the complete analysis on your dataframe
# results = run_complete_analysis(df_monorail)

# If you want to access specific results:
# error_data = results['error_summary']
# wagon_data = results['wagon_stats']
# performance_data = results['performance_comparison']

In [ ]:
results = run_complete_analysis(df_monorail)